# PandaPickCube — обучение в Google Colab

Ноутбук **самодостаточный**: не клонирует репозиторий и не вызывает `train.py` — логика взята из `config.py` и `train.py` и встроена ниже.

Стек пакетов совпадает с `environment/Dockerfile`: PyTorch (CUDA 12.4), `playground[learning]`, JAX с CUDA.

## Перед запуском

1. **Runtime → Change runtime type → GPU**.
2. Выполняйте ячейки **сверху вниз**. Установка занимает несколько минут.
3. После тяжёлой установки при странных импортах: **Runtime → Restart runtime**, затем снова ячейку с `%pip` (или пропустите, если пакеты уже стоят) и продолжайте с рабочей папки и обучением.

Чекпоинты и логи: папка `logs/` относительно текущей рабочей директории (по умолчанию `/content`). При монтировании Google Drive можно сначала выполнить `os.chdir(...)` на каталог на диске — тогда `logs/` окажется там.

## 1. Установка зависимостей

PyTorch (cu124), MuJoCo Playground с extra `learning`, JAX CUDA 12, tensorboard.

In [ ]:
%pip install -q torch --index-url https://download.pytorch.org/whl/cu124
%pip install -q "playground[learning]"
%pip install -q -U "jax[cuda12]"
%pip install -q tensorboard pytest

## 2. (Опционально) Google Drive

Раскомментируйте, смонтируйте диск, затем в следующей ячейке задайте `LOG_ROOT` на путь вроде `/content/drive/MyDrive/colab_panda`.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

## 3. Рабочая папка и переменные окружения

`MUJOCO_GL=egl` — headless-рендер на Colab. Логи пишутся в `logs/` внутри `LOG_ROOT`.

In [ ]:
import os

LOG_ROOT = "/content"  # например "/content/drive/MyDrive/colab_panda" после монтирования диска
os.makedirs(LOG_ROOT, exist_ok=True)
os.chdir(LOG_ROOT)

os.environ["MUJOCO_GL"] = "egl"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

print("cwd:", os.getcwd())

## 4. Обучение

Ниже целиком встроены `config.py` и логика из `train.py` (без `argparse`). На **T4** при OOM уменьшите `NUM_ENVS` до `1024` или `512`.

In [ ]:
"""Встроенные config.py + train.py для Colab (без внешнего репозитория)."""

import json
import os
import time
from datetime import datetime
from types import SimpleNamespace

import jax
import torch
from mujoco_playground import registry, wrapper_torch
from mujoco_playground.config import manipulation_params
from rsl_rl.runners import OnPolicyRunner

jax.config.update("jax_default_matmul_precision", "highest")

# --- Параметры запуска (аналог аргументов CLI из train.py) ---
NUM_ENVS = 4096
MAX_ITERS = 1001
SAVE_INTERVAL = 50
SEED = 1
EXP_NAME = None  # str или None → имя с датой
RESUME = None  # путь к чекпоинту или None
DEVICE = "cuda:0"

# --- config.py ---
ENV_NAME = "PandaPickCube"

ACTOR_HIDDEN_DIMS = [512, 256, 128]
CRITIC_HIDDEN_DIMS = [512, 256, 128]
ACTIVATION = "elu"
INIT_NOISE_STD = 1.0


def build_runner_cfg(obs_size) -> dict:
    cfg = manipulation_params.rsl_rl_config(ENV_NAME).to_dict()
    _policy = cfg.pop("policy", {})
    cfg["actor"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": ACTOR_HIDDEN_DIMS,
        "activation": ACTIVATION,
        "distribution_cfg": {
            "class_name": "rsl_rl.modules.distribution.GaussianDistribution",
            "init_std": INIT_NOISE_STD,
        },
    }
    cfg["critic"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": CRITIC_HIDDEN_DIMS,
        "activation": ACTIVATION,
    }
    if isinstance(obs_size, dict):
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["privileged_state"]}
    else:
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["state"]}
    return cfg


def resolve_log_dir(exp_name, resume) -> str:
    if exp_name:
        name = exp_name
    elif resume:
        name = os.path.basename(os.path.dirname(os.path.abspath(resume)))
    else:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = f"{ENV_NAME}_{timestamp}"
    log_dir = os.path.join("logs", name)
    os.makedirs(log_dir, exist_ok=True)
    return log_dir


args = SimpleNamespace(
    exp_name=EXP_NAME,
    num_envs=NUM_ENVS,
    max_iters=MAX_ITERS,
    device=DEVICE,
    resume=RESUME,
    save_interval=SAVE_INTERVAL,
    seed=SEED,
)

device = args.device
device_rank = int(device.split(":")[-1]) if "cuda" in device else 0

log_dir = resolve_log_dir(args.exp_name, args.resume)

print(f"Среда:            {ENV_NAME}")
print(f"Устройство:       {device}")
print(f"Параллельных сред: {args.num_envs}")
print(f"Итерации:         {args.max_iters}")
print(f"Лог/чекпоинты:    {log_dir}")
print()

env_cfg = registry.get_default_config(ENV_NAME)
raw_env = registry.load(ENV_NAME, config=env_cfg)

brax_env = wrapper_torch.RSLRLBraxWrapper(
    raw_env,
    num_actors=args.num_envs,
    seed=args.seed,
    episode_length=env_cfg.episode_length,
    action_repeat=1,
    device_rank=device_rank,
)
brax_env.cfg = {}

cfg_dict = build_runner_cfg(raw_env.observation_size)
cfg_dict["seed"] = args.seed
cfg_dict["max_iterations"] = args.max_iters
cfg_dict["save_interval"] = args.save_interval

config_path = os.path.join(log_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(cfg_dict, f, indent=2, default=str)
print(f"Конфиг сохранён: {config_path}")

runner = OnPolicyRunner(brax_env, cfg_dict, log_dir, device=device)

if args.resume:
    print(f"Загрузка чекпоинта: {args.resume}")
    runner.load(args.resume)

print("=" * 60)
print("  Начинаю обучение...")
print("=" * 60)
print()

t_start = time.time()
runner.learn(
    num_learning_iterations=args.max_iters,
    init_at_random_ep_len=False,
)
elapsed = time.time() - t_start

print()
print("=" * 60)
print(f"  Обучение завершено за {elapsed:.0f} с ({elapsed / 60:.1f} мин)")
print(f"  Логи и чекпоинты: {log_dir}")
print("=" * 60)

## 5. (Опционально) TensorBoard

Логи: `logs/<имя_эксперимента>/` относительно `LOG_ROOT`.

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir logs